In [1]:
import pandas as pd
import numpy as np

In [2]:
class Problem:
    def __init__(self, path="../TSPA.csv"):
        self.df = pd.read_csv(path, sep=";", header=None)
        
        cord = self.df.values[:, :-1]
        dist = np.sqrt(np.sum((cord[None, :, :] - cord[:, None, :]) ** 2, axis=2))
        
        self.dist = np.floor(dist + 0.5).astype(int)
        self.cost = self.df.values[:, -1]

    def get_n(self):
        return len(self.df)

    def get_n_nodes(self):
        # 50% of the nodes, rounded up
        return (self.get_n() + 1) // 2

    def eval(self, nodes: np.array, end=True):
        assert nodes.dtype == np.int64
        assert len(set(nodes)) == len(nodes)

        n_nodes = self.get_n_nodes()
        if end == True and len(nodes) != n_nodes:
            raise ValueError("The final path does not have the required length!")

        pairs = np.column_stack((nodes, np.roll(nodes, -1)))
        if not end:
            pairs = pairs[:-1]
        total_length = np.sum(self.dist[pairs[:, 0], pairs[:, 1]])
        total_cost = np.sum(self.cost[nodes])

        return total_length + total_cost


In [3]:
def random_solution(problem: Problem, start=None):
    n = problem.get_n()
    return np.random.choice(n, size=problem.get_n_nodes(), replace=False)

In [4]:
def nearest_neighbor_end(problem: Problem, start=None):
    n = problem.get_n()
    n_nodes = problem.get_n_nodes()
    if start is None:
        start = np.random.randint(n)

    path = [start]
    unvisited = np.ones(n, dtype=bool)
    unvisited[start] = False

    while len(path) < n_nodes:
        scores = problem.dist[path[-1]] + problem.cost
        scores = np.where(unvisited, scores, np.inf)
        nxt = int(np.argmin(scores))
        path.append(nxt)
        unvisited[nxt] = False

    return np.array(path, dtype=np.int64)

In [5]:
def nearest_neighbor_any(problem: Problem, start=None):
    n = problem.get_n()
    n_nodes = problem.get_n_nodes()
    if start is None:
        start = np.random.randint(n)

    d = problem.dist
    path = [start]
    visited = {start}

    while len(path) < n_nodes:
        best_delta, best_node, best_pos = None, None, None

        for j in range(n):
            if j in visited:
                continue

            for i in range(len(path) + 1):
                if i == 0:
                    delta = d[j, path[0]]
                elif i == len(path):
                    delta = d[path[-1], j]
                else:
                    a, b = path[i - 1], path[i]
                    delta = d[a, j] + d[j, b] - d[a, b]
                delta += problem.cost[j]

                if best_delta is None or delta < best_delta:
                    best_delta, best_node, best_pos = delta, j, i

        path.insert(best_pos, best_node)
        visited.add(best_node)

    return np.array(path, dtype=np.int64)

In [6]:
def greedy_cycle(problem: Problem, start=None):
    n = problem.get_n()
    n_nodes = problem.get_n_nodes()
    if start is None:
        start = np.random.randint(n)

    d = problem.dist

    scores = 2 * d[start] + problem.cost
    scores = np.where(np.arange(n) != start, scores, np.inf)
    second = int(np.argmin(scores))

    path = [start, second]
    visited = {start, second}

    while len(path) < n_nodes:
        best_delta, best_node, best_pos = None, None, None

        for j in range(n):
            if j in visited:
                continue

            for i in range(len(path)):
                a, b = path[i - 1], path[i]
                delta = d[a, j] + d[j, b] - d[a, b] + problem.cost[j]

                if best_delta is None or delta < best_delta:
                    best_delta, best_node, best_pos = delta, j, i

        path.insert(best_pos, best_node)
        visited.add(best_node)

    return np.array(path, dtype=np.int64)

In [7]:
def eval(f: callable, p: Problem, iter=None):
    # by default one run per node, the i-th run starts from node i
    if iter is None:
        iter = p.get_n()

    values = []
    best = None
    best_val = float("inf")

    for i in range(iter):
        solution = f(p, start=i % p.get_n())
        val = p.eval(solution)
        values.append(val)
        if val < best_val:
            best = solution
            best_val = val
    
    values = np.array(values)

    print(f"Min: {np.min(values)}")
    print(f"Max: {np.max(values)}")
    print(f"Mean: {np.mean(values)}")

    return best, values

In [8]:
import os

methods = {
    "random": random_solution,
    "nearest_neighbor_end": nearest_neighbor_end,
    "nearest_neighbor_any": nearest_neighbor_any,
    "greedy_cycle": greedy_cycle,
}

np.random.seed(0)
os.makedirs("../data", exist_ok=True)
problems = {name: Problem(f"../{name}.csv") for name in ["TSPA", "TSPB"]}

results = {}
best_solutions = {}
for name, p in problems.items():
    for method, f in methods.items():
        print(f"{name} / {method}")
        best, values = eval(f, p)
        results[(method, name)] = values
        best_solutions[(method, name)] = best
        # read by visualize.py
        pd.DataFrame({"solution": best}).to_parquet(f"../data/{method}_{name.lower()}")

TSPA / random
Min: 238989
Max: 290456
Mean: 262874.39


TSPA / nearest_neighbor_end


Min: 84362
Max: 90075
Mean: 87476.63
TSPA / nearest_neighbor_any


Min: 72328
Max: 77022
Mean: 74346.965
TSPA / greedy_cycle


Min: 72590
Max: 75345
Mean: 73792.87
TSPB / random
Min: 185119
Max: 225987
Mean: 206009.685
TSPB / nearest_neighbor_end


Min: 56007
Max: 63166
Mean: 59397.335
TSPB / nearest_neighbor_any


Min: 52915
Max: 66164
Mean: 59475.495
TSPB / greedy_cycle


Min: 55655
Max: 63736
Mean: 59777.5


In [9]:
rows = []
for method in methods:
    row = {"Method": method}
    for name in problems:
        v = results[(method, name)]
        row[f"{name} min"] = v.min()
        row[f"{name} max"] = v.max()
        row[f"{name} mean"] = round(v.mean(), 2)
    rows.append(row)
summary = pd.DataFrame(rows)
summary

,Method,TSPA min,TSPA max,TSPA mean,TSPB min,TSPB max,TSPB mean
0,random,238989,290456,262874.39,185119,225987,206009.68
1,nearest_neighbor_end,84362,90075,87476.63,56007,63166,59397.34
2,nearest_neighbor_any,72328,77022,74346.96,52915,66164,59475.50
3,greedy_cycle,72590,75345,73792.87,55655,63736,59777.50


In [10]:
# independent check of the best solutions: the right number of distinct nodes
# and the objective recomputed from scratch with plain loops
for (method, name), s in best_solutions.items():
    p = problems[name]
    nodes = [int(x) for x in s]
    assert len(nodes) == p.get_n_nodes()
    assert len(set(nodes)) == len(nodes)
    assert all(0 <= x < p.get_n() for x in nodes)

    length = sum(p.dist[nodes[i], nodes[(i + 1) % len(nodes)]] for i in range(len(nodes)))
    cost = sum(p.cost[x] for x in nodes)
    assert length + cost == results[(method, name)].min()
print("All best solutions are valid")

All best solutions are valid


In [11]:
for method in methods:
    for name in problems:
        s = best_solutions[(method, name)]
        print(f"{method} / {name} ({problems[name].eval(s)}):")
        print(", ".join(str(x) for x in s))
        print()

random / TSPA (238989):
190, 101, 51, 7, 77, 40, 198, 196, 156, 191, 131, 183, 23, 90, 72, 119, 187, 31, 14, 55, 103, 99, 132, 194, 116, 63, 180, 6, 89, 1, 166, 48, 33, 174, 57, 68, 59, 158, 113, 149, 45, 39, 9, 24, 159, 37, 96, 121, 104, 83, 129, 141, 91, 41, 102, 70, 74, 8, 10, 176, 155, 182, 73, 125, 120, 79, 130, 146, 78, 28, 143, 150, 20, 66, 2, 52, 98, 153, 161, 123, 92, 151, 4, 177, 56, 17, 188, 133, 25, 32, 71, 11, 162, 139, 34, 124, 135, 164, 171, 46

random / TSPB (185119):
148, 152, 128, 95, 61, 29, 193, 118, 69, 106, 140, 112, 177, 160, 26, 8, 77, 85, 34, 147, 72, 135, 86, 21, 142, 40, 133, 92, 175, 11, 139, 84, 88, 51, 178, 98, 60, 57, 22, 74, 32, 64, 70, 186, 53, 30, 121, 157, 156, 96, 107, 19, 110, 82, 90, 155, 3, 97, 168, 101, 33, 130, 174, 188, 185, 166, 138, 50, 195, 189, 49, 115, 16, 108, 109, 27, 12, 183, 54, 59, 184, 191, 120, 52, 80, 199, 39, 179, 159, 162, 151, 111, 154, 119, 35, 150, 17, 198, 7, 143

nearest_neighbor_end / TSPA (84362):
54, 9, 33, 36, 177, 67, 1